# Solar AI — ground-PV CONDITION research detector, ACTIVE 4-CLASS taxonomy (YOLO11n)

Phase 15B.1 — **TRAINING-DURATION ABLATION** of the Phase 15B baseline
(`condition_yolo11n_4class_baseline_v1`, checkpoint
`weights/yolo_solar_condition_4class_research_v2.pt`: test P=.592 R=.440 mAP50=.444
mAP50-95=.236, best_epoch=97/100, mAP still rising at the epoch-100 cutoff).

**ONLY epochs (100→200) and patience (20→40) change here. Everything else — dataset,
base model, imgsz, seed, optimizer, cos_lr, augmentation — is identical to v2.** This run
exists solely to answer: was v2 under-trained, or does more training not materially help?

**Active 4-class second-stage condition model:** `0=CLEAN 1=DUSTY 2=BIRD_DROP 3=PHYSICAL_DAMAGE`.

- **Experimental research model only.** NOT the panel detector. Does NOT touch / replace
  the production aerial detector `weights/yolo_solar.pt`, the ground panel detector
  `weights/yolo_solar_ground_research_v1.pt`, or any FROZEN prior condition checkpoint
  (`weights/yolo_solar_condition_research_v1.pt` [5-class],
  `weights/yolo_solar_condition_4class_research_v1.pt` [Phase 15A CPU smoke], or
  `weights/yolo_solar_condition_4class_research_v2.pt` [Phase 15B 100-epoch baseline]).
- Base model: pretrained **YOLO11n** COCO checkpoint (never a Solar AI model).
- Dataset: `edithstark/solar-ai-ground-condition-4class-research-v1` (PRIVATE, SAME dataset
  as v2 — no data changes in this ablation).

## ⚠️ PROVENANCE — grade C, UNVERIFIED
The underlying images ultimately trace to Kaggle `pythonafroz/solar-panel-images`
(licence **Unknown**). Tag: `personal-experimental-unverified-provenance`. Do NOT
redistribute / publish / use commercially / claim verified CC BY image rights.

## ⚠️ ELECTRICAL_DAMAGE is NOT part of this taxonomy
It existed in the prior 5-class dataset; those annotations visually correspond to cell
cracks, not confirmed electrical failure, so they were removed (never relabelled as
PHYSICAL_DAMAGE or anything else). This model cannot and does not predict it.

## ⚠️ HOTSPOT requires thermal/IR data
Never inferred from RGB. Out of scope for this or any RGB condition model.

## ⚠️ PHYSICAL_DAMAGE is the rarest active class
314 images / 725 instances vs. DUSTY's 1,364 images / 8,550 instances. Reported
prominently and separately below — aggregate mAP must not hide this.

imgsz=640, epochs=**200**, patience=**40**, seed=42, cos_lr, optimizer=auto — ONE
controlled duration ablation, not a hyperparameter sweep. Attach the dataset in the
sidebar and set Accelerator = GPU before running.


## 1. Environment


In [ ]:
import sys, platform, subprocess, json, os, time, hashlib, shutil
from pathlib import Path
print('Python :', platform.python_version())


### CUDA — fail closed if unavailable
Kaggle's free GPU pool can hand out a Pascal **Tesla P100 (sm_60)**. Pin
`torch==2.7.1` / `torchvision==0.22.1` (CUDA 12.6 wheels, last line retaining Pascal
support) — the pins proven on this account's prior ground-detector / condition runs.


In [ ]:
PINNED = ['torch==2.7.1', 'torchvision==0.22.1', 'ultralytics==8.4.138', 'PyYAML>=6.0.1']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINNED], check=True)

import torch
assert torch.cuda.is_available(), 'CUDA unavailable — set Settings -> Accelerator -> GPU.'
import torchvision, ultralytics
ENV = dict(
    python=platform.python_version(), torch=torch.__version__,
    torchvision=torchvision.__version__, cuda=torch.version.cuda,
    ultralytics=ultralytics.__version__,
    gpu=torch.cuda.get_device_name(0),
    gpu_capability='.'.join(map(str, torch.cuda.get_device_capability(0))),
    gpu_memory_gb=round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1),
)
print(json.dumps(ENV, indent=2))


## 2. Locate the attached Kaggle dataset + verify provenance/hashes


In [ ]:
candidates = list(Path('/kaggle/input').rglob('data.yaml'))
assert candidates, 'No data.yaml under /kaggle/input — attach the dataset in the sidebar.'
SRC_DATA_YAML = None
for c in candidates:
    if '4class' in str(c).lower():
        SRC_DATA_YAML = c; break
SRC_DATA_YAML = SRC_DATA_YAML or candidates[0]
DATA_ROOT = SRC_DATA_YAML.parent
print('mounted data.yaml :', SRC_DATA_YAML)
print(SRC_DATA_YAML.read_text())

import yaml as _yaml
_cfg = _yaml.safe_load(SRC_DATA_YAML.read_text())
_cfg['path'] = str(DATA_ROOT.resolve())
_cfg['train'], _cfg['val'], _cfg['test'] = 'images/train', 'images/valid', 'images/test'
DATA_YAML = Path('/kaggle/working/ground_condition_4class_data.yaml')
DATA_YAML.write_text(_yaml.safe_dump(_cfg, sort_keys=False))
print('training data.yaml (absolute):'); print(DATA_YAML.read_text())
for _s in ('images/train','images/valid','images/test','labels/train','labels/valid','labels/test'):
    assert (DATA_ROOT / _s).is_dir(), f'missing {_s} under {DATA_ROOT}'
print('all split dirs present')

PROV = DATA_ROOT / 'provenance'
assert PROV.is_dir(), 'no provenance/ directory in the dataset'
pkg_summary = json.loads((DATA_ROOT / 'PACKAGE_SUMMARY.json').read_text())
print('provenance_grade :', pkg_summary['provenance_grade'])
print('provenance_tag   :', pkg_summary['provenance_tag'])
print('electrical_damage_removed :', pkg_summary['electrical_damage_removed'])
assert pkg_summary['provenance_grade'] == 'C'
assert pkg_summary['provenance_tag'] == 'personal-experimental-unverified-provenance'
assert pkg_summary['electrical_damage_removed'] is True
assert pkg_summary['no_family_crosses_splits'] is True
assert pkg_summary['no_exact_duplicate_crosses_splits'] is True
assert pkg_summary['counts'] == {'train': 1686, 'valid': 483, 'test': 242}, pkg_summary['counts']

def _sha_file(p):
    h = hashlib.sha256(); h.update(Path(p).read_bytes()); return h.hexdigest()
SPLIT_CSV = PROV / 'splits_4class_manifest.csv'
DATASET_HASHES = dict(
    split_manifest_sha256=_sha_file(SPLIT_CSV),
    package_dataset_content_sha256=pkg_summary['dataset_content_sha256'],
    package_data_yaml_sha256=pkg_summary['data_yaml_sha256'],
)
print(json.dumps(DATASET_HASHES, indent=2))
assert DATASET_HASHES['split_manifest_sha256'] == pkg_summary['split_manifest_sha256']

# Ablation guard: the dataset must be BYTE-IDENTICAL to the v2 baseline's dataset —
# this is a training-duration ablation, not a data change.
V2_DATASET_HASHES = dict(
    split_manifest_sha256='600752137a4cbf8b24f506339b662ea0d0dbb67d1fc897f52fef83d6eb1d893e',
    package_dataset_content_sha256='6637e6361343e7bb17d1703e705bffc0cb733a5428f48ce681fa28bc8826bf92',
    package_data_yaml_sha256='0731819484bb3b07c87a2ced8e79997d380fa5a34a4701fdb3845c8fa03cffe4',
)
assert DATASET_HASHES == V2_DATASET_HASHES, f'dataset differs from v2! {DATASET_HASHES} != {V2_DATASET_HASHES}'
print('dataset hashes match v2 baseline exactly — this is a pure training-duration ablation')


## 3. Validate the ACTIVE 4-class taxonomy + per-split, per-class statistics (no training yet)


In [ ]:
import yaml, csv
from collections import Counter, defaultdict
cfg = yaml.safe_load(DATA_YAML.read_text())
names = cfg['names']
if isinstance(names, list): names = {i: n for i, n in enumerate(names)}
NAMES = {int(k): v for k, v in names.items()}
EXPECTED = {0: 'CLEAN', 1: 'DUSTY', 2: 'BIRD_DROP', 3: 'PHYSICAL_DAMAGE'}
assert NAMES == EXPECTED, f'taxonomy mismatch: {NAMES} != {EXPECTED}'
assert 'ELECTRICAL_DAMAGE' not in NAMES.values(), 'ELECTRICAL_DAMAGE must not be present'
assert not any('snow' in v.lower() for v in NAMES.values())
assert 'panel' not in [v.lower() for v in NAMES.values()], 'PANEL must not be a class'
print('taxonomy OK (active 4-class, ELECTRICAL_DAMAGE absent):', NAMES)

rows = list(csv.DictReader(open(SPLIT_CSV, encoding='utf-8')))
fam_by_split_class = defaultdict(lambda: defaultdict(set))
for r in rows:
    s = r['derived_split']
    for c in r['unified_classes'].split(';'):
        if c and c != '(none)':
            fam_by_split_class[s][c].add(r['family_id'])

STATS = {}
for split_key in ('train', 'val', 'test'):
    rel = cfg[split_key]; sname = {'train':'train','val':'valid','test':'test'}[split_key]
    img_dir = DATA_ROOT / rel
    lbl_dir = DATA_ROOT / rel.replace('images/', 'labels/')
    imgs = [p for p in img_dir.glob('*') if p.suffix.lower() in {'.jpg','.jpeg','.png','.webp'}]
    inst, img_with = Counter(), Counter()
    for im in imgs:
        lp = lbl_dir / (im.stem + '.txt')
        if not lp.is_file(): continue
        seen = set()
        for line in lp.read_text().splitlines():
            p = line.split()
            if len(p) == 5:
                c = EXPECTED[int(p[0])]; inst[c] += 1; seen.add(c)
        for c in seen: img_with[c] += 1
    STATS[sname] = dict(
        images=len(imgs), instances=dict(inst), images_with_class=dict(img_with),
        unique_families_per_class={c: len(f) for c, f in fam_by_split_class[sname].items()},
    )
    print(f'  {sname:5s} images={len(imgs):5d} instances={dict(inst)}')
    print(f'        images_with={dict(img_with)}  fam={STATS[sname]["unique_families_per_class"]}')


## 4. Train `condition_yolo11n_4class_200ep_v1` — the DURATION ABLATION
Base `yolo11n.pt` (COCO-pretrained), SAME as v2. **Only `epochs` (100→200) and
`patience` (20→40) differ from v2** — everything else below is byte-identical to the
v2 baseline's train_args.


In [ ]:
from ultralytics import YOLO
PROJECT = '/kaggle/working/research_runs/ground_condition_4class'
RUN_NAME = 'condition_yolo11n_4class_200ep_v1'
BASE_MODEL = 'yolo11n.pt'
SEED = 42

train_args = dict(
    data=str(DATA_YAML), epochs=200, imgsz=640, patience=40, seed=SEED,
    batch=-1, device=0, project=PROJECT, name=RUN_NAME, exist_ok=True,
    pretrained=True, cos_lr=True, optimizer='auto', verbose=True, val=True,
    hsv_h=0.015, hsv_s=0.4, hsv_v=0.4, degrees=5.0, translate=0.1, scale=0.3,
    shear=0.0, perspective=0.0, flipud=0.0, fliplr=0.5, mosaic=0.5, mixup=0.0, copy_paste=0.0,
)
print(json.dumps({k: str(v) for k, v in train_args.items()}, indent=2))

t0 = time.time()
model = YOLO(BASE_MODEL)
results = model.train(**train_args)
train_seconds = time.time() - t0
RUN_DIR = Path(results.save_dir)
print('run dir:', RUN_DIR, ' elapsed(s):', round(train_seconds))


## 5. Validation-only confidence THRESHOLD ANALYSIS (test set untouched here)
Evaluate a confidence range on **validation only**. The test set is not touched in this
cell — it is evaluated exactly once, afterward, at the chosen operating threshold.


In [ ]:
best = RUN_DIR / 'weights' / 'best.pt'
assert best.is_file(), best
eval_model = YOLO(str(best))

THRESHOLDS = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]
threshold_analysis = []
for conf in THRESHOLDS:
    m = eval_model.val(data=str(DATA_YAML), split='val', conf=conf, imgsz=640, iou=0.5,
                        plots=False, verbose=False)
    box = m.box
    per_class = {}
    for pos, cidx in enumerate(list(box.ap_class_index)):
        p, r, ap50, ap = box.class_result(pos)
        per_class[EXPECTED[int(cidx)]] = dict(precision=float(p), recall=float(r))
    row = dict(conf=conf, precision=float(box.mp), recall=float(box.mr),
               mAP50=float(box.map50), mAP50_95=float(box.map), per_class=per_class)
    threshold_analysis.append(row)
    print(f"conf={conf:.2f}  P={row['precision']:.3f}  R={row['recall']:.3f}  "
          f"mAP50={row['mAP50']:.3f}  PHYSICAL_DAMAGE_R={per_class.get('PHYSICAL_DAMAGE',{}).get('recall','n/a')}")

# Pick an operating threshold: the smallest conf in the sweep whose overall F1
# (harmonic mean of P/R) is within 2% of the sweep's best F1 — prefers a lower/more-
# permissive threshold over the single sharpest peak, since a research 'assistive' model
# should not be over-tuned to one operating point. Never touches the test set to choose this.
def _f1(p, r): return (2 * p * r / (p + r)) if (p + r) > 0 else 0.0
f1s = [_f1(row['precision'], row['recall']) for row in threshold_analysis]
best_f1 = max(f1s)
candidates = [row for row, f1 in zip(threshold_analysis, f1s) if f1 >= best_f1 * 0.98]
OPERATING_THRESHOLD = min(c['conf'] for c in candidates)
print('\nchosen OPERATING_THRESHOLD (validation-only selection):', OPERATING_THRESHOLD)

# explicit report: is F1 still monotonically increasing toward the lowest tested value?
monotonic_toward_low = all(f1s[i] >= f1s[i+1] for i in range(len(f1s)-1))
print('F1 monotonically decreasing as conf rises (best F1 at the LOWEST tested threshold):',
      monotonic_toward_low, ' — if True, the true optimum may lie below', THRESHOLDS[0])


## 6. Evaluate the FINAL selected checkpoint — held-out TEST set, evaluated ONCE, at the operating threshold


In [ ]:
def evaluate(split, conf):
    m = eval_model.val(data=str(DATA_YAML), split=split, conf=conf, imgsz=640, iou=0.5,
                        plots=True, verbose=False)
    box = m.box
    per_class = {}
    for pos, cidx in enumerate(list(box.ap_class_index)):
        p, r, ap50, ap = box.class_result(pos)
        per_class[EXPECTED[int(cidx)]] = dict(precision=float(p), recall=float(r),
                                              AP50=float(ap50), AP50_95=float(ap))
    return dict(
        overall=dict(precision=float(box.mp), recall=float(box.mr),
                     mAP50=float(box.map50), mAP50_95=float(box.map)),
        per_class=per_class, save_dir=str(m.save_dir), conf=conf)

evaluation = {}
evaluation['val_default'] = evaluate('val', 0.001)
evaluation['val_operating'] = evaluate('val', OPERATING_THRESHOLD)
evaluation['test_default'] = evaluate('test', 0.001)
evaluation['test_operating'] = evaluate('test', OPERATING_THRESHOLD)

for k, v in evaluation.items():
    print(f'\n== {k} (conf={v["conf"]}) =='); print(json.dumps(v['overall'], indent=2))
    print('per-class:', json.dumps(v['per_class'], indent=2))

for c, d in evaluation['test_operating']['per_class'].items():
    d['test_images_with_class'] = STATS['test']['images_with_class'].get(c, 0)
    d['test_instances'] = STATS['test']['instances'].get(c, 0)
    d['test_unique_families'] = STATS['test']['unique_families_per_class'].get(c, 0)

print('\n*** RARE-CLASS FOCUS (never let aggregate mAP hide these) ***')
for c in ('PHYSICAL_DAMAGE', 'BIRD_DROP'):
    d = evaluation['test_operating']['per_class'].get(c, {})
    print(f"{c}: recall={d.get('recall')}  AP50={d.get('AP50')}  AP50_95={d.get('AP50_95')}  "
          f"test_instances={d.get('test_instances')}  test_images={d.get('test_images_with_class')}")

print('\n*** v3 vs v2 (baseline) DELTA ***')
V2_TEST_DEFAULT = dict(precision=0.5920913154902189, recall=0.44047422200046077,
                       mAP50=0.4437736214675794, mAP50_95=0.2360894573262115)
v3 = evaluation['test_default']['overall']
for k in ('precision', 'recall', 'mAP50', 'mAP50_95'):
    print(f'  {k}: v2={V2_TEST_DEFAULT[k]:.4f}  v3={v3[k]:.4f}  delta={v3[k]-V2_TEST_DEFAULT[k]:+.4f}')


## 7. Package artifacts + reproducibility manifest + SHA-256
Records every field Phase 15B/15B.1 requires: architecture, dataset hashes, imgsz/epochs/
patience/batch/optimizer/seed, augmentation, environment (Python/torch/torchvision/
Ultralytics/CUDA/GPU), training duration, best epoch, and both the default and
operating-threshold metrics, plus the explicit v2-vs-v3 delta.


In [ ]:
OUT = Path('/kaggle/working/output'); OUT.mkdir(exist_ok=True)
for name in ('best.pt', 'last.pt'):
    src = RUN_DIR / 'weights' / name
    if src.is_file(): shutil.copy2(src, OUT / name)
for pat in ('results.csv','results.png','args.yaml',
            'confusion_matrix.png','confusion_matrix_normalized.png',
            'BoxPR_curve.png','BoxF1_curve.png','BoxP_curve.png','BoxR_curve.png',
            'PR_curve.png','F1_curve.png','P_curve.png','R_curve.png',
            'labels.jpg','labels_correlogram.jpg',
            'train_batch*.jpg','val_batch*.jpg'):
    for f in RUN_DIR.rglob(pat): shutil.copy2(f, OUT / f.name)
for sd in {e['save_dir'] for e in evaluation.values()}:
    sd = Path(sd)
    if sd.is_dir(): shutil.copytree(sd, OUT / ('eval_' + sd.name), dirs_exist_ok=True)

pred_dir = OUT / 'predictions'; pred_dir.mkdir(exist_ok=True)
test_imgs = sorted((DATA_ROOT / 'images/test').glob('*'))[:24]
if test_imgs:
    YOLO(str(best)).predict(source=[str(p) for p in test_imgs], conf=OPERATING_THRESHOLD, save=True,
                            project=str(pred_dir), name='test_examples', exist_ok=True)

sha = hashlib.sha256((OUT / 'best.pt').read_bytes()).hexdigest()
last_sha = hashlib.sha256((OUT / 'last.pt').read_bytes()).hexdigest() if (OUT/'last.pt').is_file() else None

import csv as _csv
results_rows = list(_csv.DictReader(open(RUN_DIR / 'results.csv')))
best_epoch = None
if results_rows:
    key = next((k for k in results_rows[0] if 'mAP50-95' in k and 'B' in k), None)
    if key:
        best_epoch = max(range(len(results_rows)), key=lambda i: float(results_rows[i][key])) + 1

manifest = dict(
    experiment_name=RUN_NAME,
    ablation_of='condition_yolo11n_4class_baseline_v1 (weights/yolo_solar_condition_4class_research_v2.pt) — TRAINING DURATION ONLY (epochs 100->200, patience 20->40); dataset/architecture/augmentation/optimizer unchanged',
    model_role='EXPERIMENTAL RESEARCH condition detector, ACTIVE 4-CLASS taxonomy (second stage, on panel crops) — NOT a panel detector, NOT production, NOT validated for field use',
    active_taxonomy=EXPECTED,
    electrical_damage_status='REMOVED from the active taxonomy entirely — not present in this dataset, never relabelled',
    hotspot_status='OUT OF SCOPE — requires thermal/IR data, never inferred from RGB',
    dataset_kaggle='edithstark/solar-ai-ground-condition-4class-research-v1',
    dataset_derived_from='the existing 5-class condition dataset, ELECTRICAL_DAMAGE removed',
    provenance_grade='C', provenance_tag='personal-experimental-unverified-provenance',
    provenance_warning='underlying images trace to Kaggle pythonafroz/solar-panel-images (licence Unknown); do NOT redistribute/publish/use commercially/claim verified CC BY',
    rare_class_warning='PHYSICAL_DAMAGE is the rarest active class (314 train+val+test images / 725 instances) — see per_class["test_operating"]["PHYSICAL_DAMAGE"] before trusting aggregate mAP',
    base_model=BASE_MODEL, pretrained_source='YOLO11n COCO-pretrained (Ultralytics)',
    epochs=train_args['epochs'], imgsz=train_args['imgsz'], batch=train_args['batch'],
    patience=train_args['patience'], optimizer='auto', cos_lr=True, seed=SEED,
    augmentation={k: train_args[k] for k in ('hsv_h','hsv_s','hsv_v','degrees','translate',
        'scale','shear','perspective','flipud','fliplr','mosaic','mixup','copy_paste')},
    full_train_args={k: str(v) for k, v in train_args.items()},
    device='cuda:0', env=ENV,
    dataset_hashes=DATASET_HASHES,
    dataset_stats=STATS,
    train_seconds=train_seconds, best_epoch=best_epoch, total_epochs=train_args['epochs'],
    run_dir=str(RUN_DIR),
    threshold_analysis=threshold_analysis, operating_threshold=OPERATING_THRESHOLD,
    f1_monotonic_toward_lowest_threshold=monotonic_toward_low,
    val_default_metrics=evaluation['val_default']['overall'],
    val_operating_metrics=evaluation['val_operating']['overall'],
    test_default_metrics=evaluation['test_default']['overall'],
    test_operating_metrics=evaluation['test_operating']['overall'],
    per_class=dict(val_default=evaluation['val_default']['per_class'],
                   val_operating=evaluation['val_operating']['per_class'],
                   test_default=evaluation['test_default']['per_class'],
                   test_operating=evaluation['test_operating']['per_class']),
    checkpoint_sha256=sha, last_checkpoint_sha256=last_sha,
    not_production_validated=True,
)
(OUT / 'training_manifest.json').write_text(json.dumps(manifest, indent=2, default=str))
(OUT / 'evaluation.json').write_text(json.dumps(evaluation, indent=2, default=str))
(OUT / 'threshold_analysis.json').write_text(json.dumps(threshold_analysis, indent=2, default=str))
(OUT / 'environment.json').write_text(json.dumps(ENV, indent=2))
print('best.pt sha256:', sha)
print(json.dumps(manifest, indent=2, default=str))
print('\nArtifacts:', sorted(p.name for p in OUT.iterdir()))


## 8. Retrieve
Download `/kaggle/working/output/`. Locally save `best.pt` as
`weights/yolo_solar_condition_4class_research_v3.pt` (gitignored research artifact) —
**never** overwrite `weights/yolo_solar.pt`, `weights/yolo_solar_ground_research_v1.pt`,
`weights/yolo_solar_condition_research_v1.pt` (5-class, frozen),
`weights/yolo_solar_condition_4class_research_v1.pt` (Phase 15A CPU smoke, frozen), or
`weights/yolo_solar_condition_4class_research_v2.pt` (Phase 15B 100-epoch baseline, frozen).

This condition model is a research artifact only, NOT production-validated. It is a
**second-stage** model that runs on panel crops (Phase 14A's dynamic panel detector feeds
it). It makes no RGB-hotspot claim and cannot predict ELECTRICAL_DAMAGE (removed from the
active taxonomy). A missed panel upstream cannot be classified by this model — condition-
model metrics and end-to-end pipeline reliability are separate claims.
